# XGBoost training

Build a unified training frame with `build_training_frame()`: static IEEE columns from `fraud.bronze.train_features`, operational columns from silver transactions/identities, and rolling behavioral features computed on the fly from silver history.

**Note:** Run the install cell first so `fraud_scoring_engine` is available for this session.


In [0]:
# Install fraud_scoring_engine package for this session
%pip install -q /Workspace/Users/yannickkh@outlook.com/fraud-scoring-engine


In [0]:
from fraud_scoring_engine.training import build_training_frame, time_split


In [0]:
TARGET = "is_fraud"
LIMIT = 10_000


In [0]:
df = build_training_frame(spark, limit=LIMIT)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")


In [0]:
display(df.shape)
display(df[TARGET].value_counts())
display(df[["transaction_id", "transaction_amt", "card1", "product_cd", "velocity_1h", TARGET]].head())


## Train / validation / test split

Use `time_split()` to cut the frame into contiguous time blocks so later transactions cannot leak into training. Fit preprocessing on train only.


In [0]:
split = time_split(df)

for name, frame in [("train", split.train), ("val", split.val), ("test", split.test)]:
    fraud_rate = frame[TARGET].mean()
    print(
        f"{name:5s}  rows={len(frame):5,}  "
        f"fraud={int(frame[TARGET].sum()):4d}  "
        f"rate={fraud_rate:.4f}  "
        f"dt=[{frame['transaction_dt'].min()}, {frame['transaction_dt'].max()}]"
    )


## Feature preprocessing

Use `FraudFeaturePreprocessor` to drop metadata columns, keep numeric nulls for XGBoost, and encode categoricals with a stable `__MISSING__` level. Fit on train only so validation and test vocabularies cannot leak.


In [0]:
from fraud_scoring_engine.preprocessing import FraudFeaturePreprocessor

preprocessor = FraudFeaturePreprocessor()
X_train = preprocessor.fit_transform(split.train)
X_val = preprocessor.transform(split.val)
X_test = preprocessor.transform(split.test)

y_train = split.train[TARGET]
y_val = split.val[TARGET]
y_test = split.test[TARGET]

print(X_train.shape, X_val.shape, X_test.shape)
print("Number of categorical features:", X_train.select_dtypes("category").shape[1])
print("Number of float features:", X_train.select_dtypes("float").shape[1])
print("Number of int features:", X_train.select_dtypes("int").shape[1])


## MLflow tracking with Unity Catalog

Log **validation** metrics during search; log **test** metrics only on the final candidate run. All models logged with signatures and input examples for deployment readiness.


In [0]:
from pathlib import Path
import logging
import warnings

# Configure logging BEFORE importing MLflow to suppress noisy warnings
logging.getLogger('mlflow.tracking.context.registry').setLevel(logging.ERROR)  # Py4J warnings
logging.getLogger('py4j.security').setLevel(logging.ERROR)
logging.getLogger('mlflow.sklearn').setLevel(logging.ERROR)  # Preprocessor artifact warnings
logging.getLogger('mlflow.models.model').setLevel(logging.WARNING)  # Suppress INFO about preprocessor signature
warnings.filterwarnings('ignore', message='.*Inferred schema contains integer column.*')
warnings.filterwarnings('ignore', message='.*Failed to validate serving input example.*')

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import optuna
import xgboost as xgb
from mlflow.models import infer_signature
from sklearn.metrics import average_precision_score, roc_auc_score

from fraud_scoring_engine.config import get_mlflow_tracking_uri
from fraud_scoring_engine.training import setup_mlflow

EXPERIMENT_NAME = "/Users/yannickkh@outlook.com/fraud.experiments.xgboost_fraud"
N_TRIALS = 15

# Using Unity Catalog for experiments
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_NAME)

# class-imbalance weight
scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())

DATA_INFO = {
    "limit": LIMIT,
    "split_ratios": "0.70/0.15/0.15",
    "time_col": "transaction_dt",
    "n_train": len(X_train),
    "n_val": len(X_val),
    "n_test": len(X_test),
    "n_features": X_train.shape[1],
    "scale_pos_weight": scale_pos_weight,
}

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Registry: databricks-uc (Unity Catalog)")
print(f"scale_pos_weight: {scale_pos_weight:.4f}")


def evaluate_probs(y_true, y_proba: object) -> dict[str, float]:
    return {
        "pr_auc": float(average_precision_score(y_true, y_proba)),
        "roc_auc": float(roc_auc_score(y_true, y_proba)),
    }


def fit_xgb(params: dict, *, verbose: bool = False) -> xgb.XGBClassifier:
    model = xgb.XGBClassifier(
        enable_categorical=True,
        tree_method="hist",
        eval_metric="aucpr",
        early_stopping_rounds=30,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        **params,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=verbose)
    return model


def log_model_artifacts(model: xgb.XGBClassifier, fitted_preprocessor, X_preprocessed) -> str:
    """Log model with signature for deployment readiness.
    
    Args:
        model: Fitted XGBoost model
        fitted_preprocessor: Fitted preprocessor (logged separately)
        X_preprocessed: Preprocessed feature matrix (X_train, X_val, or X_test)
    
    Returns:
        model_uri: URI of the logged model for later registration
    """
    # Create signature from preprocessed features (what the model expects)
    signature_input = X_preprocessed.head(100)
    predictions = model.predict_proba(signature_input)[:, 1]
    signature = infer_signature(signature_input, predictions)
    
    # Log model with signature only (skip input_example to avoid serialization warnings)
    # The signature alone is sufficient for Unity Catalog registration and Model Serving
    model_info = mlflow.xgboost.log_model(
        model,
        name="model",
        signature=signature
    )
    
    # Log preprocessor separately for transparency
    mlflow.sklearn.log_model(
        fitted_preprocessor,
        name="preprocessor",
        skops_trusted_types=[
            "builtins.frozenset",
            "fraud_scoring_engine.preprocessing.preprocessor.FraudFeaturePreprocessor",
        ],
    )
    
    # Log feature names for reference
    feature_names_path = Path("feature_names.txt")
    feature_names_path.write_text(
        "\n".join(fitted_preprocessor.get_feature_names_out()),
        encoding="utf-8",
    )
    mlflow.log_artifact(str(feature_names_path))
    feature_names_path.unlink(missing_ok=True)
    
    return model_info.model_uri


## Baseline XGBoost

Train a fixed-parameter model with early stopping on validation. Logs val PR-AUC / ROC-AUC only (no test).


In [0]:
BASELINE_PARAMS = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 1.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
}

with mlflow.start_run(run_name="xgboost-baseline") as baseline_run:
    mlflow.set_tag("stage", "baseline")
    mlflow.log_params(DATA_INFO)
    mlflow.log_params(BASELINE_PARAMS)

    baseline_model = fit_xgb(BASELINE_PARAMS)
    val_metrics = evaluate_probs(y_val, baseline_model.predict_proba(X_val)[:, 1])
    mlflow.log_metrics(
        {
            "val_pr_auc": val_metrics["pr_auc"],
            "val_roc_auc": val_metrics["roc_auc"],
            "best_iteration": int(getattr(baseline_model, "best_iteration", -1)),
        }
    )
    # Log with signature and input example for deployment readiness
    model_uri = log_model_artifacts(baseline_model, preprocessor, X_train)
    
    print(f"run_id: {baseline_run.info.run_id}")
    print(f"model_uri: {model_uri}")
    print(f"val PR-AUC:  {val_metrics['pr_auc']:.4f}")
    print(f"val ROC-AUC: {val_metrics['roc_auc']:.4f}")
    print(f"best_iteration: {getattr(baseline_model, 'best_iteration', None)}")


## Hyperparameter search (Optuna + nested MLflow runs)

Parent run owns the study; each trial is a nested child that logs params and **val** PR-AUC. Objective maximizes validation PR-AUC.


In [0]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

with mlflow.start_run(run_name="xgboost-optuna") as parent_run:
    mlflow.set_tag("stage", "tuning")
    mlflow.log_params(DATA_INFO)
    mlflow.log_param("n_trials", N_TRIALS)

    def objective(trial: optuna.Trial) -> float:
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
        }

        with mlflow.start_run(nested=True, run_name=f"trial-{trial.number}"):
            mlflow.log_params(params)
            model = fit_xgb(params)
            metrics = evaluate_probs(y_val, model.predict_proba(X_val)[:, 1])
            mlflow.log_metrics(
                {
                    "val_pr_auc": metrics["pr_auc"],
                    "val_roc_auc": metrics["roc_auc"],
                    "best_iteration": int(getattr(model, "best_iteration", -1)),
                }
            )
            trial.set_user_attr("best_iteration", int(getattr(model, "best_iteration", -1)))
            return metrics["pr_auc"]

    study = optuna.create_study(direction="maximize", study_name="xgboost-fraud")
    study.optimize(objective, n_trials=N_TRIALS)

    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metrics({"best_val_pr_auc": float(study.best_value)})

    print(f"parent run_id: {parent_run.info.run_id}")
    print(f"best val PR-AUC: {study.best_value:.4f}")
    print(f"best params: {study.best_params}")


## Candidate evaluation (test once)

Retrain with the best Optuna params, log **test** metrics, and tag `stage=candidate`.


In [0]:
best_params = study.best_params

with mlflow.start_run(run_name="xgboost-candidate") as candidate_run:
    mlflow.set_tag("stage", "candidate")
    mlflow.log_params(DATA_INFO)
    mlflow.log_params(best_params)
    mlflow.log_param("tuning_parent_run_id", parent_run.info.run_id)

    candidate_model = fit_xgb(best_params)
    val_metrics = evaluate_probs(y_val, candidate_model.predict_proba(X_val)[:, 1])
    test_metrics = evaluate_probs(y_test, candidate_model.predict_proba(X_test)[:, 1])

    mlflow.log_metrics(
        {
            "val_pr_auc": val_metrics["pr_auc"],
            "val_roc_auc": val_metrics["roc_auc"],
            "test_pr_auc": test_metrics["pr_auc"],
            "test_roc_auc": test_metrics["roc_auc"],
            "best_iteration": int(getattr(candidate_model, "best_iteration", -1)),
        }
    )
    # Log with signature and input example for deployment readiness
    model_uri = log_model_artifacts(candidate_model, preprocessor, X_train)
    
    print(f"run_id: {candidate_run.info.run_id}")
    print(f"model_uri: {model_uri}")
    print(f"\nMetrics:")
    print(f"  val  PR-AUC: {val_metrics['pr_auc']:.4f}  ROC-AUC: {val_metrics['roc_auc']:.4f}")
    print(f"  test PR-AUC: {test_metrics['pr_auc']:.4f}  ROC-AUC: {test_metrics['roc_auc']:.4f}")
    print(f"\n✓ Model ready for Unity Catalog registration")
